# Freedom Investment Challenge — ML-блок

**Часть A. Модель «справедливого мультипликатора» (отбор недооценённых компаний).**
Нейросеть (обучение backpropagation) учится на ~800 компаниях S&P 500 и S&P 400 предсказывать, какой EV/EBITDA компании положен по её росту, марже, рентабельности, долгу, риску и сектору. Если реальный мультипликатор сильно ниже предсказанного, а бизнес качественный — компания кандидат в недооценённые.

Чтобы результат был честным:
* прогноз для каждой компании делает модель, которая эту компанию **не видела** (кросс-валидация на 5 частей);
* нейросеть сравнивается с простыми методами: медианой сектора, линейной регрессией и градиентным бустингом;
* цена акции не используется как признак (иначе модель «подглядывает» в ответ): размер компании берём по выручке, а не по капитализации.

**Часть B. Прогноз волатильности (GARCH против EWMA и исторической) и матрица корреляций** — для распределения $100 000. Результат вставляется в Excel-модель.

**Как запустить:** Среда выполнения → Выполнить все. Время: 15–25 минут (основное — загрузка данных с Yahoo). Банки и недвижимость в модель не входят: для них EV/EBITDA не имеет смысла.

In [ ]:
%pip install -q -U yfinance

In [ ]:
# ===== Настройки =====
DEMO = False                 # True — синтетические данные (проверка кода без интернета)
PRICE_DATE = "2026-11-02"    # дата фиксации цен по условиям конкурса
INCLUDE_MIDCAP = True        # добавить S&P 400 (компании $2–20 млрд — менее изученные)
MIN_MCAP = 2e9               # правило конкурса: капитализация ≥ $2 млрд
MIN_DOLLAR_VOL = 3e6         # правило конкурса: среднедневной оборот за 3 мес. ≥ $3 млн
MIN_LISTING_YEARS = 3        # правило конкурса: на бирже ≥ 3 лет
TOP_N = 30                   # сколько кандидатов показать
USE_CACHE = True             # повторный запуск берёт уже скачанные данные из CSV
SEED = 42
OUR_FIVE = ["ADBE", "GOOGL", "T", "SYK", "BAC"]  # текущая пятёрка (порядок как в Excel)
FINAL = OUR_FIVE             # для части B: замените, если пятёрка изменится

In [ ]:
import io, os, time, math, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
np.random.seed(SEED)
pd.set_option("display.width", 200, "display.max_columns", 30)
PRICE_TS = pd.Timestamp(PRICE_DATE)
if not DEMO:
    import requests, yfinance as yf

## A1. Список компаний и загрузка данных

In [ ]:
FIELDS = ["shortName", "sector", "industry", "quoteType", "marketCap", "enterpriseValue", "ebitda", "totalRevenue",
          "revenueGrowth", "earningsGrowth", "grossMargins", "ebitdaMargins", "operatingMargins", "profitMargins",
          "returnOnEquity", "returnOnAssets", "totalDebt", "totalCash", "freeCashflow", "operatingCashflow", "beta",
          "sharesOutstanding", "currentPrice", "forwardPE", "trailingPE", "enterpriseToEbitda",
          "firstTradeDateEpochUtc", "firstTradeDateMilliseconds", "fiftyTwoWeekHigh", "fiftyTwoWeekLow",
          "shortPercentOfFloat", "recommendationMean", "targetMeanPrice"]


def get_universe():
    hdr = {"User-Agent": "Mozilla/5.0 (research project)"}
    urls = {"S&P 500": "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
            "S&P 400": "https://en.wikipedia.org/wiki/List_of_S%26P_400_companies"}
    frames = []
    for name, url in urls.items():
        if name == "S&P 400" and not INCLUDE_MIDCAP:
            continue
        try:
            tables = pd.read_html(io.StringIO(requests.get(url, headers=hdr, timeout=30).text))
            for tb in tables:
                cols = [c for c in tb.columns if str(c).lower().startswith(("symbol", "ticker"))]
                if cols and len(tb) > 300:
                    frames.append(pd.DataFrame({"ticker": tb[cols[0]].astype(str).str.strip(), "index": name}))
                    break
        except Exception as e:
            print("Не удалось загрузить", name, "—", e)
    if not frames:
        t = pd.read_csv("https://raw.githubusercontent.com/datasets/s-and-p-500-companies/main/data/constituents.csv")
        frames.append(pd.DataFrame({"ticker": t["Symbol"], "index": "S&P 500"}))
    u = pd.concat(frames).drop_duplicates("ticker").reset_index(drop=True)
    u["yf"] = u["ticker"].str.replace(".", "-", regex=False)
    for t in OUR_FIVE:                      # наша пятёрка всегда в выборке
        if t not in set(u["yf"]):
            u.loc[len(u)] = [t, "наша пятёрка", t]
    return u


def fetch_info(sym, tries=4):
    for k in range(tries):
        try:
            info = yf.Ticker(sym).info or {}
            if info.get("quoteType") is None and info.get("marketCap") is None:
                raise ValueError("пустой ответ")
            row = {f: info.get(f) for f in FIELDS}
            row["yf"] = sym
            return row
        except Exception:
            time.sleep(5 * (k + 1))     # Yahoo ограничивает частоту запросов — ждём и пробуем снова
    return {"yf": sym}


def get_fundamentals(universe):
    path = "fundamentals_raw.csv"
    if USE_CACHE and os.path.exists(path):
        print("Беру фундаментальные данные из кэша", path)
        return pd.read_csv(path)
    rows = []
    with ThreadPoolExecutor(max_workers=4) as ex:
        futs = {ex.submit(fetch_info, s): s for s in universe["yf"]}
        for i, f in enumerate(as_completed(futs), 1):
            rows.append(f.result())
            if i % 100 == 0:
                print(f"  загружено {i}/{len(futs)}")
    df = pd.DataFrame(rows)
    failed = df.loc[df["marketCap"].isna(), "yf"].tolist() if "marketCap" in df else list(df["yf"])
    if failed:
        print(f"Повторная загрузка {len(failed)} тикеров по одному...")
        time.sleep(30)
        again = pd.DataFrame([fetch_info(s) for s in failed])
        df = pd.concat([df[~df["yf"].isin(failed)], again], ignore_index=True)
    print("Без данных осталось:", int(df["marketCap"].isna().sum()))
    df.to_csv(path, index=False)
    return df


def get_prices(symbols, start):
    end = (PRICE_TS + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    data = yf.download(list(symbols), start=start, end=end, auto_adjust=False, progress=False, threads=True,
                       group_by="column")
    close, vol = data["Close"], data["Volume"]
    if isinstance(close, pd.Series):
        close, vol = close.to_frame(symbols[0]), vol.to_frame(symbols[0])
    return close, vol

In [ ]:
def make_demo_data(n=750, seed=SEED):
    # Синтетика с той же структурой, что у Yahoo: нужна только для проверки кода.
    rng = np.random.default_rng(seed)
    sectors = ["Technology", "Healthcare", "Industrials", "Consumer Cyclical", "Consumer Defensive",
               "Communication Services", "Energy", "Utilities", "Basic Materials", "Financial Services", "Real Estate"]
    syms = [f"D{i:03d}" for i in range(n - len(OUR_FIVE))] + OUR_FIVE
    sec = rng.choice(sectors, n)
    sec[-5:] = ["Technology", "Communication Services", "Communication Services", "Healthcare", "Financial Services"]
    rev = np.exp(rng.normal(22.5, 1.2, n))
    g = np.clip(rng.normal(0.07, 0.10, n), -0.4, 0.8)
    m = np.clip(rng.normal(0.22, 0.10, n), 0.02, 0.6)
    roe = np.clip(rng.normal(0.18, 0.12, n), -0.5, 0.9)
    lev = np.clip(rng.normal(1.8, 1.3, n), -2, 7)
    beta = np.clip(rng.normal(1.0, 0.35, n), 0.2, 2.5)
    sec_eff = {s: rng.normal(0, 0.15) for s in sectors}
    logm = 2.3 + 2.2 * g + 1.2 * m + 0.5 * roe - 0.08 * lev - 0.15 * (beta - 1) + np.array([sec_eff[s] for s in sec]) \
        + 0.6 * np.tanh(4 * g) * m + rng.normal(0, 0.22, n)
    ebitda = rev * m
    ev = np.exp(logm) * ebitda
    netdebt = lev * ebitda
    mcap = ev - netdebt
    price = np.exp(rng.normal(4.2, 0.8, n))
    shares = mcap / price
    df = pd.DataFrame({
        "yf": syms, "shortName": [f"Demo {s}" for s in syms], "sector": sec,
        "industry": np.where(rng.random(n) < 0.02, "Tobacco", "Demo industry"), "quoteType": "EQUITY",
        "marketCap": mcap, "enterpriseValue": ev, "ebitda": ebitda, "totalRevenue": rev, "revenueGrowth": g,
        "earningsGrowth": g * 1.5 + rng.normal(0, 0.1, n), "grossMargins": np.clip(m + 0.25, 0, 0.95),
        "ebitdaMargins": m, "operatingMargins": m * 0.8, "profitMargins": m * 0.55, "returnOnEquity": roe,
        "returnOnAssets": roe / 2.5, "totalDebt": np.maximum(netdebt, 0) + 0.2 * ebitda, "totalCash": 0.2 * ebitda,
        "freeCashflow": rev * (m * 0.6 + rng.normal(0, 0.03, n)), "operatingCashflow": rev * m * 0.8, "beta": beta,
        "sharesOutstanding": shares, "currentPrice": price, "forwardPE": np.nan, "trailingPE": np.nan,
        "enterpriseToEbitda": ev / ebitda, "firstTradeDateEpochUtc": np.where(rng.random(n) < 0.03, 1.72e9, 9.0e8),
        "firstTradeDateMilliseconds": np.nan, "fiftyTwoWeekHigh": price * 1.3, "fiftyTwoWeekLow": price * 0.8,
        "shortPercentOfFloat": 0.03, "recommendationMean": 2.2, "targetMeanPrice": price * 1.12})
    dates = pd.bdate_range(end=PRICE_TS, periods=1900)
    close, vol = {}, {}
    for i, s in enumerate(syms):
        T = len(dates); sig = np.empty(T); r = np.empty(T); sig[0] = 0.018
        for t in range(T):
            if t:
                sig[t] = np.sqrt(0.000004 + 0.08 * r[t - 1] ** 2 + 0.9 * sig[t - 1] ** 2)
            r[t] = sig[t] * rng.standard_t(6) / np.sqrt(1.5)
        close[s] = price[i] * np.exp(np.cumsum(r) - r.sum())
        vol[s] = np.exp(rng.normal(14, 1, T))
    return df, pd.DataFrame(close, index=dates), pd.DataFrame(vol, index=dates)


if DEMO:
    raw, close_all, vol_all = make_demo_data()
    universe = pd.DataFrame({"yf": raw["yf"], "index": "demo"})
else:
    universe = get_universe()
    print("Компаний в списке:", len(universe))
    raw = get_fundamentals(universe)
    close_all, vol_all = get_prices(universe["yf"].tolist(), start=(PRICE_TS - pd.DateOffset(years=1, days=10)).strftime("%Y-%m-%d"))
print("Фундаментальные данные:", raw.shape, " Цены:", close_all.shape)

## A2. Признаки

In [ ]:
df = universe[["yf", "index"]].merge(raw, on="yf", how="left")

# Цена на дату фиксации, оборот за 3 месяца, падение от максимума
px_hist = close_all.loc[:PRICE_TS]
price_date_used = px_hist.index.max()
if price_date_used.normalize() != PRICE_TS:
    print(f"ВНИМАНИЕ: последней доступной датой цены является {price_date_used.date()}, а не {PRICE_DATE}. "
          "Перезапустите после закрытия торгов 2 ноября.")
last = px_hist.ffill().iloc[-1]
dollar_vol = (px_hist * vol_all.loc[:PRICE_TS]).tail(63).mean()
hi52 = px_hist.tail(252).max()
ret3m = px_hist.ffill().iloc[-1] / px_hist.ffill().iloc[-64] - 1
df["price"] = df["yf"].map(last)
df["dollar_vol_3m"] = df["yf"].map(dollar_vol)
df["drawdown_52w"] = df["yf"].map(last / hi52 - 1)
df["ret_3m"] = df["yf"].map(ret3m)

# Капитализация и EV на дату фиксации: чистый долг из Yahoo, цена — на 02.11
df["net_debt"] = df["enterpriseValue"] - df["marketCap"]
# Капитализация по всем классам акций (у GOOGL, FOX, NWS и др. sharesOutstanding — только один класс)
df["shares_total"] = (df["marketCap"] / df["currentPrice"]).where(
    df["marketCap"].notna() & (df["currentPrice"] > 0), df["sharesOutstanding"])
df["mcap"] = df["price"] * df["shares_total"]
df["ev"] = df["mcap"] + df["net_debt"]
df["ev_ebitda"] = df["ev"] / df["ebitda"]

ft = df["firstTradeDateEpochUtc"].fillna(df["firstTradeDateMilliseconds"] / 1000)
df["listed_since"] = pd.to_datetime(ft, unit="s", errors="coerce")

clip = lambda s, lo, hi: s.astype(float).clip(lo, hi)
df["f_rev_growth"] = clip(df["revenueGrowth"], -0.5, 1.0)
df["f_earn_growth"] = clip(df["earningsGrowth"], -1.0, 2.0)
df["f_gross_margin"] = clip(df["grossMargins"], -0.5, 1.0)
df["f_ebitda_margin"] = clip(df["ebitdaMargins"], -0.5, 1.0)
df["f_op_margin"] = clip(df["operatingMargins"], -0.5, 1.0)
df["f_roe"] = clip(df["returnOnEquity"], -1.0, 1.5)
df["f_roa"] = clip(df["returnOnAssets"], -0.5, 0.5)
df["f_fcf_margin"] = clip(df["freeCashflow"] / df["totalRevenue"], -0.5, 0.8)
df["f_net_debt_ebitda"] = clip(df["net_debt"] / df["ebitda"], -5, 10)
df["f_beta"] = clip(df["beta"], 0, 3)
df["f_log_revenue"] = np.log(df["totalRevenue"].where(df["totalRevenue"] > 0))
NUM = [c for c in df.columns if c.startswith("f_")]
CAT = ["sector"]
NICE = {"f_rev_growth": "Рост выручки", "f_earn_growth": "Рост прибыли", "f_gross_margin": "Валовая маржа",
        "f_ebitda_margin": "Маржа EBITDA", "f_op_margin": "Операционная маржа", "f_roe": "ROE", "f_roa": "ROA",
        "f_fcf_margin": "Маржа FCF", "f_net_debt_ebitda": "Долг / EBITDA", "f_beta": "Бета",
        "f_log_revenue": "Размер (выручка)", "sector": "Сектор"}

EXCLUDED_SECTORS = ["Financial Services", "Real Estate"]
model_mask = (~df["sector"].isin(EXCLUDED_SECTORS) & (df["ebitda"] > 0) & (df["ev"] > 0)
              & df["ev_ebitda"].between(2, 80) & df["price"].notna() & df["sector"].notna())
data = df[model_mask].copy().reset_index(drop=True)
data["y"] = np.log(data["ev_ebitda"])
X, y = data[NUM + CAT], data["y"].values
print(f"Для обучения: {len(data)} компаний (исключены банки/страховщики/недвижимость, убыточные по EBITDA и выбросы)")

## A3. Модели: медиана сектора, линейная регрессия, бустинг, нейросеть (backpropagation)

In [ ]:
def make_pre():
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)])


class SectorMedian(BaseEstimator, RegressorMixin):
    # Простейший ориентир: «компания стоит как медиана своего сектора»
    def fit(self, X, y):
        s = pd.Series(np.asarray(y), index=X.index)
        self.med_ = s.groupby(X["sector"]).median().to_dict()
        self.all_ = float(np.median(y))
        return self

    def predict(self, X):
        return X["sector"].map(self.med_).fillna(self.all_).values


class Net:
    # Полносвязная сеть: вход → ReLU → ReLU → выход. Прямой и обратный проход (backpropagation) написаны вручную.
    def __init__(self, sizes, rng):
        self.W = [rng.normal(0, math.sqrt(2 / a), (a, b)) for a, b in zip(sizes[:-1], sizes[1:])]  # инициализация He
        self.b = [np.zeros(b) for b in sizes[1:]]

    def forward(self, X, dropout=0.0, rng=None):
        self.cache = [X]
        a = X
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = a @ W + b
            if i < len(self.W) - 1:                       # скрытые слои
                mask = (z > 0).astype(float)
                if dropout > 0:
                    mask *= (rng.random(z.shape) >= dropout) / (1 - dropout)
                a = z * mask
                self.cache.append(mask)
            else:
                a = z
            self.cache.append(a)
        return a

    def backward(self, d_out):
        # d_out = ∂L/∂(выход). Идём от последнего слоя к первому по цепному правилу.
        grads_W, grads_b = [None] * len(self.W), [None] * len(self.W)
        delta = d_out
        for i in reversed(range(len(self.W))):
            a_prev = self.cache[2 * i]
            grads_W[i] = a_prev.T @ delta
            grads_b[i] = delta.sum(0)
            if i > 0:
                delta = (delta @ self.W[i].T) * self.cache[2 * i - 1]   # производная ReLU (и маска dropout)
        return grads_W, grads_b


def mse_and_grad(net, X, y, l2, dropout=0.0, rng=None):
    out = net.forward(X, dropout, rng)
    err = out - y
    loss = (err ** 2).mean() + l2 * sum((W ** 2).sum() for W in net.W)
    gW, gb = net.backward(2 * err / len(y))
    gW = [g + 2 * l2 * W for g, W in zip(gW, net.W)]
    return loss, gW, gb


def gradient_check(seed=0):
    # Сравниваем градиенты backprop с численной производной: относительная ошибка должна быть < 1e-6
    rng = np.random.default_rng(seed)
    net = Net([5, 4, 3, 1], rng)
    X, y = rng.normal(size=(8, 5)), rng.normal(size=(8, 1))
    _, gW, _ = mse_and_grad(net, X, y, l2=1e-3)
    worst = 0.0
    for k in range(len(net.W)):
        for idx in np.ndindex(net.W[k].shape):
            old = net.W[k][idx]
            net.W[k][idx] = old + 1e-6; lp, _, _ = mse_and_grad(net, X, y, 1e-3)
            net.W[k][idx] = old - 1e-6; lm, _, _ = mse_and_grad(net, X, y, 1e-3)
            net.W[k][idx] = old
            num = (lp - lm) / 2e-6
            worst = max(worst, abs(num - gW[k][idx]) / max(1e-8, abs(num) + abs(gW[k][idx])))
    return worst


class NumpyMLP(BaseEstimator, RegressorMixin):
    # Обучение: мини-батчи, Adam, L2-регуляризация, dropout, ранняя остановка; среднее из нескольких запусков
    def __init__(self, hidden=(64, 32), dropout=0.1, lr=1e-3, l2=1e-4, epochs=600, patience=40,
                 batch=64, n_seeds=3, seed=SEED):
        self.hidden, self.dropout, self.lr, self.l2 = hidden, dropout, lr, l2
        self.epochs, self.patience, self.batch, self.n_seeds, self.seed = epochs, patience, batch, n_seeds, seed

    def fit(self, X, y):
        self.pre_ = make_pre().fit(X)
        Xt = self.pre_.transform(X)
        y = np.asarray(y, dtype=float)
        self.mu_, self.sd_ = y.mean(), y.std() + 1e-9
        yt = ((y - self.mu_) / self.sd_).reshape(-1, 1)
        self.nets_, self.curves_ = [], []
        for k in range(self.n_seeds):
            rng = np.random.default_rng(self.seed + k)
            perm = rng.permutation(len(Xt))
            n_val = max(1, int(0.15 * len(Xt)))
            vi, ti = perm[:n_val], perm[n_val:]
            net = Net([Xt.shape[1], *self.hidden, 1], rng)
            params = net.W + net.b
            m = [np.zeros_like(p) for p in params]
            v = [np.zeros_like(p) for p in params]
            t, b1, b2 = 0, 0.9, 0.999
            best, best_params, bad, curve = np.inf, None, 0, []
            for ep in range(self.epochs):
                order = rng.permutation(ti)
                for s in range(0, len(order), self.batch):
                    idx = order[s:s + self.batch]
                    _, gW, gb = mse_and_grad(net, Xt[idx], yt[idx], self.l2, self.dropout, rng)
                    t += 1
                    for j, (p, g) in enumerate(zip(net.W + net.b, gW + gb)):   # шаг Adam
                        m[j] = b1 * m[j] + (1 - b1) * g
                        v[j] = b2 * v[j] + (1 - b2) * g * g
                        p -= self.lr * (m[j] / (1 - b1 ** t)) / (np.sqrt(v[j] / (1 - b2 ** t)) + 1e-8)
                vl = float(((net.forward(Xt[vi]) - yt[vi]) ** 2).mean())
                curve.append(vl)
                if vl < best - 1e-5:
                    best, bad = vl, 0
                    best_params = ([W.copy() for W in net.W], [b.copy() for b in net.b])
                else:
                    bad += 1
                    if bad >= self.patience:
                        break
            net.W, net.b = best_params
            self.nets_.append(net)
            self.curves_.append(curve)
        return self

    def predict(self, X):
        Xt = self.pre_.transform(X)
        p = np.mean([n.forward(Xt) for n in self.nets_], axis=0).ravel()
        return p * self.sd_ + self.mu_


print(f"Проверка градиентов backprop: максимальная относительная ошибка = {gradient_check():.2e}")


class Ensemble(BaseEstimator, RegressorMixin):
    def __init__(self, members=None):
        self.members = members

    def fit(self, X, y):
        self.fitted_ = [clone(m).fit(X, y) for m in self.members]
        return self

    def predict(self, X):
        return np.mean([m.predict(X) for m in self.fitted_], axis=0)


MODELS = {
    "Медиана сектора": SectorMedian(),
    "Линейная регрессия (Ridge)": Pipeline([("pre", make_pre()), ("m", RidgeCV(alphas=np.logspace(-2, 3, 20)))]),
    "Градиентный бустинг": Pipeline([("pre", make_pre()), ("m", HistGradientBoostingRegressor(
        max_iter=400, learning_rate=0.04, max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
        random_state=SEED))]),
    "Нейросеть (MLP, backprop)": NumpyMLP(),
}

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof = {name: np.zeros(len(y)) for name in MODELS}
for fold, (tr, te) in enumerate(kf.split(X), 1):
    for name, model in MODELS.items():
        m = clone(model).fit(X.iloc[tr], y[tr])
        oof[name][te] = m.predict(X.iloc[te])
    print(f"Часть {fold}/5 готова")
oof["Ансамбль (Ridge + бустинг + MLP)"] = np.mean(
    [oof["Линейная регрессия (Ridge)"], oof["Градиентный бустинг"], oof["Нейросеть (MLP, backprop)"]], axis=0)

quality = pd.DataFrame({
    "Модель": list(oof),
    "R² (на невиденных компаниях)": [r2_score(y, p) for p in oof.values()],
    "Средняя ошибка (лог)": [mean_absolute_error(y, p) for p in oof.values()],
}).set_index("Модель")
quality["Типичная ошибка мультипликатора, %"] = (np.exp(quality["Средняя ошибка (лог)"]) - 1) * 100
quality = quality.sort_values("Средняя ошибка (лог)")
BEST = quality.index[0]
print(quality.round(3))
print("\nЛучшая модель:", BEST)

## A4. Справедливая цена и фильтры конкурса

In [ ]:
data["fair_ev_ebitda"] = np.exp(oof[BEST])
data["fair_ev"] = data["fair_ev_ebitda"] * data["ebitda"]
data["fair_price"] = (data["fair_ev"] - data["net_debt"]) / data["shares_total"]
data["upside"] = data["fair_price"] / data["price"] - 1
data["discount_mult"] = data["ev_ebitda"] / data["fair_ev_ebitda"] - 1
data["analyst_upside"] = data["targetMeanPrice"] / data["price"] - 1

BANNED = ["tobacco", "winer", "distill", "brewer", "gambling", "casino", "coal", "aerospace & defense"]
ind = data["industry"].fillna("").str.lower()
data["banned"] = ind.apply(lambda s: any(k in s for k in BANNED))
cutoff = PRICE_TS - pd.DateOffset(years=MIN_LISTING_YEARS)
data["rule_mcap"] = data["mcap"] >= MIN_MCAP
data["rule_volume"] = data["dollar_vol_3m"] >= MIN_DOLLAR_VOL
data["rule_listing"] = data["listed_since"].isna() | (data["listed_since"] <= cutoff)
data["rule_equity"] = data["quoteType"].fillna("EQUITY").eq("EQUITY")
data["passes_rules"] = data[["rule_mcap", "rule_volume", "rule_listing", "rule_equity"]].all(axis=1) & ~data["banned"]

# Проверка на «ловушку стоимости»: дешёвая заслуженно?
data["trap_fcf"] = data["freeCashflow"] <= 0
data["trap_debt"] = data["f_net_debt_ebitda"] > 4
data["trap_shrinking"] = data["f_rev_growth"] < -0.05
data["quality_ok"] = ~(data["trap_fcf"] | data["trap_debt"] | data["trap_shrinking"])

funnel = pd.DataFrame([
    ("Компании S&P 500" + (" + S&P 400" if INCLUDE_MIDCAP else ""), len(df)),
    ("С данными, EBITDA > 0, не банки и не недвижимость", len(data)),
    ("Проходят правила конкурса", int(data["passes_rules"].sum())),
    ("Без признаков ловушки стоимости", int((data["passes_rules"] & data["quality_ok"]).sum())),
    ("Дешевле справедливого мультипликатора на 20%+", int((data["passes_rules"] & data["quality_ok"]
                                                           & (data["discount_mult"] <= -0.2)).sum())),
], columns=["Этап", "Компаний"])
print(funnel.to_string(index=False))

cols = ["yf", "shortName", "sector", "price", "ev_ebitda", "fair_ev_ebitda", "discount_mult", "upside",
        "f_rev_growth", "f_ebitda_margin", "f_fcf_margin", "f_net_debt_ebitda", "drawdown_52w", "ret_3m",
        "analyst_upside", "mcap"]
cand = (data[data["passes_rules"] & data["quality_ok"]].sort_values("discount_mult").head(TOP_N)[cols]
        .rename(columns={"yf": "Тикер", "shortName": "Компания", "sector": "Сектор", "price": "Цена",
                         "ev_ebitda": "EV/EBITDA факт", "fair_ev_ebitda": "EV/EBITDA справедл.",
                         "discount_mult": "Дисконт мультипл.", "upside": "Потенциал по модели",
                         "f_rev_growth": "Рост выручки", "f_ebitda_margin": "Маржа EBITDA",
                         "f_fcf_margin": "Маржа FCF", "f_net_debt_ebitda": "Долг/EBITDA",
                         "drawdown_52w": "От макс. 52 нед.", "ret_3m": "За 3 мес.",
                         "analyst_upside": "Потенциал по аналитикам", "mcap": "Капитализация"}))
cand.round(3)

In [ ]:
# Наша пятёрка глазами модели
rows = []
for t in OUR_FIVE:
    r = data[data["yf"] == t]
    if len(r):
        r = r.iloc[0]
        rank = int((data.loc[data["passes_rules"], "discount_mult"] < r["discount_mult"]).sum()) + 1
        rows.append([t, r["price"], r["ev_ebitda"], r["fair_ev_ebitda"], r["discount_mult"], r["upside"], rank,
                     "да" if r["quality_ok"] else "нет", "Модель применима"])
    else:
        sec = df.loc[df["yf"] == t, "sector"]
        why = ("Банк/страховщик — оценивается по P/TBV, EV/EBITDA не применим"
               if len(sec) and sec.iloc[0] in EXCLUDED_SECTORS else "Нет данных или выброс")
        rows.append([t] + [np.nan] * 6 + ["", why])
ours = pd.DataFrame(rows, columns=["Тикер", "Цена", "EV/EBITDA факт", "EV/EBITDA справедл.", "Дисконт мультипл.",
                                   "Потенциал по модели", "Место по дисконту", "Качество ок", "Комментарий"])
print(f"Место — среди {int(data['passes_rules'].sum())} компаний, прошедших правила (1 = самая дешёвая)")
ours.round(3)

## A5. Что сильнее всего влияет на справедливый мультипликатор

In [ ]:
best_est = (Ensemble([MODELS["Линейная регрессия (Ridge)"], MODELS["Градиентный бустинг"],
                      MODELS["Нейросеть (MLP, backprop)"]]) if BEST.startswith("Ансамбль") else clone(MODELS[BEST]))
best_est.fit(X, y)
pi = permutation_importance(best_est, X, y, scoring="neg_mean_absolute_error", n_repeats=5, random_state=SEED)
imp = (pd.Series(pi.importances_mean, index=[NICE[c] for c in X.columns]).sort_values(ascending=True))

os.makedirs("charts", exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 5))
imp.plot.barh(ax=ax, color="#2B7A4B")
ax.set_title(f"Важность признаков ({BEST})")
ax.set_xlabel("Рост ошибки при перемешивании признака")
plt.tight_layout(); plt.savefig("charts/feature_importance.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(np.exp(oof[BEST]), data["ev_ebitda"], s=10, alpha=0.35, color="#9AA5A0", label="Компании")
top = data["yf"].isin(cand["Тикер"].head(10))
ax.scatter(np.exp(oof[BEST])[top], data.loc[top, "ev_ebitda"], s=30, color="#2B7A4B", label="Топ-10 кандидатов")
mine = data["yf"].isin(OUR_FIVE)
ax.scatter(np.exp(oof[BEST])[mine], data.loc[mine, "ev_ebitda"], s=60, color="#D9822B", marker="D", label="Наша пятёрка")
for _, r in data[mine].iterrows():
    ax.annotate(r["yf"], (r["fair_ev_ebitda"], r["ev_ebitda"]), xytext=(4, 4), textcoords="offset points")
lim = [2, 60]
ax.plot(lim, lim, "--", color="#555", lw=1)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lim); ax.set_ylim(lim)
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_locator(FixedLocator([2, 3, 4, 6, 8, 10, 15, 20, 30, 40, 60]))
    axis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}x"))
    axis.set_minor_locator(NullLocator())
ax.set_xlabel("Справедливый EV/EBITDA (прогноз модели)"); ax.set_ylabel("Фактический EV/EBITDA")
ax.set_title("Ниже диагонали — дешевле, чем положено по качеству бизнеса")
ax.legend(); plt.tight_layout(); plt.savefig("charts/fair_vs_actual.png", dpi=200); plt.show()

fig, ax = plt.subplots(figsize=(7, 3.8))
q = quality["Типичная ошибка мультипликатора, %"].sort_values()
q.plot.barh(ax=ax, color=["#2B7A4B" if i == BEST else "#9AA5A0" for i in q.index])
ax.set_xlabel("Типичная ошибка на невиденных компаниях, %"); ax.set_title("Сравнение моделей")
plt.tight_layout(); plt.savefig("charts/model_comparison.png", dpi=200); plt.show()

## B. Волатильность и корреляции для распределения $100 000
Сравниваем три способа прогнозировать волатильность на следующий месяц — историческая (63 дня), EWMA (λ = 0,94) и GARCH(1,1) — на данных 2025–2026 гг., которые модель не видела при каждом прогнозе. Лучший метод даёт прогноз волатильности на 12 месяцев. Корреляции — по дневным доходностям за 2 года со сжатием Ledoit-Wolf.

In [ ]:
def garch_fit(r):
    # GARCH(1,1) методом максимального правдоподобия; σ² долгосрочная = выборочная дисперсия (variance targeting)
    e = (r - r.mean()).values
    s2 = e.var()

    def neg_ll(p):
        a, b = p
        if a < 0 or b < 0 or a + b >= 0.999:
            return 1e10
        w = s2 * (1 - a - b)
        h = np.empty_like(e); h[0] = s2
        for t in range(1, len(e)):
            h[t] = w + a * e[t - 1] ** 2 + b * h[t - 1]
        return 0.5 * np.sum(np.log(h) + e ** 2 / h)

    best = min((minimize(neg_ll, x0, method="Nelder-Mead", options={"xatol": 1e-5, "fatol": 1e-4, "maxiter": 400})
                for x0 in ([0.05, 0.90], [0.10, 0.80])), key=lambda res: res.fun)
    a, b = best.x
    w = s2 * (1 - a - b)
    h = s2
    for t in range(1, len(e)):
        h = w + a * e[t - 1] ** 2 + b * h
    h_next = w + a * e[-1] ** 2 + b * h
    return a, b, s2, h_next


def garch_path(r, h):
    a, b, s2, h1 = garch_fit(r)
    k = np.arange(h)
    return s2 + (a + b) ** k * (h1 - s2)    # прогноз дисперсии на 1..h дней вперёд


if DEMO:
    px = close_all[FINAL].loc[:PRICE_TS]
else:
    px, _ = get_prices(FINAL, start="2019-01-01")
    px = px[FINAL]
ret = (100 * np.log(px).diff()).dropna(how="all")


def ewma_var(r, lam=0.94):
    v = r.var()
    for x in r.values:
        v = lam * v + (1 - lam) * x * x
    return v


H = 21
eval_dates = [d for d in ret.loc["2025-01-01":].resample("ME").last().index if d <= ret.index[-1 - H]]
recs = []
for t in FINAL:
    r = ret[t].dropna()
    for d in eval_dates:
        hist = r.loc[:d].tail(1000)
        fut = r.loc[d:].iloc[1:H + 1]
        if len(hist) < 300 or len(fut) < H:
            continue
        realized = (fut ** 2).sum()
        fc = {"Историческая 63 дня": hist.tail(63).var() * H,
              "EWMA": ewma_var(hist.tail(500)) * H,
              "GARCH(1,1)": garch_path(hist, H).sum()}
        for k, v in fc.items():
            recs.append([t, d, k, v, realized])
ev = pd.DataFrame(recs, columns=["Тикер", "Дата", "Метод", "Прогноз", "Факт"])
ev["QLIKE"] = ev["Факт"] / ev["Прогноз"] - np.log(ev["Факт"] / ev["Прогноз"]) - 1
ev["Ошибка, % волат."] = (np.sqrt(ev["Прогноз"]) - np.sqrt(ev["Факт"])).abs() / np.sqrt(ev["Факт"]) * 100
vol_quality = ev.groupby("Метод")[["QLIKE", "Ошибка, % волат."]].mean().sort_values("QLIKE")
BEST_VOL = vol_quality.index[0]
print(vol_quality.round(3)); print("\nЛучший метод:", BEST_VOL)

vol12 = {}
for t in FINAL:
    r = ret[t].dropna().tail(1000)
    if BEST_VOL == "GARCH(1,1)":
        var_year = garch_path(r, 252).sum()
    elif BEST_VOL == "EWMA":
        var_year = ewma_var(r.tail(500)) * 252
    else:
        var_year = r.tail(63).var() * 252
    vol12[t] = math.sqrt(var_year) / 100
vol12 = pd.Series(vol12, name="Прогноз волатильности на 12 мес.")
hist_vol = (ret.tail(252).std() * math.sqrt(252) / 100).rename("Историческая за год")

lw = LedoitWolf().fit(ret.tail(504).dropna().values)
cov = lw.covariance_
sd = np.sqrt(np.diag(cov))
corr = pd.DataFrame(cov / np.outer(sd, sd), index=FINAL, columns=FINAL)
avg_corr = corr.values[np.triu_indices(len(FINAL), 1)].mean()
print("\nСредняя корреляция:", round(avg_corr, 3))
corr.round(2)

In [ ]:
# Веса: обратно волатильности (как в Excel) и минимальная дисперсия с полной матрицей корреляций, лимиты 10–30%
inv = (1 / vol12) / (1 / vol12).sum()
S = np.outer(vol12.values, vol12.values) * corr.values
cons = [{"type": "eq", "fun": lambda w: w.sum() - 1}]
mv = minimize(lambda w: w @ S @ w, np.full(len(FINAL), 1 / len(FINAL)), bounds=[(0.10, 0.30)] * len(FINAL),
              constraints=cons, method="SLSQP").x
port = pd.DataFrame({"Историческая за год": hist_vol, "Прогноз на 12 мес.": vol12,
                     "Вес: обратно риску": inv, "Вес: мин. дисперсия": mv}, index=FINAL)
for name in ["Вес: обратно риску", "Вес: мин. дисперсия"]:
    w = port[name].values
    print(f"{name}: волатильность портфеля {math.sqrt(w @ S @ w):.1%}")
port.round(3)

## Сохранение результатов

In [ ]:
paste = pd.DataFrame({
    "Что": [f"Волатильность {t}" for t in FINAL] + ["Средняя корреляция"],
    "Значение": list(vol12.values) + [avg_corr],
    "Куда вставить в Excel-модель": [f"'Вводные'!I{11 + i}" for i in range(len(FINAL))] + ["'Портфель'!C5"],
})
with pd.ExcelWriter("ml_results.xlsx") as w:
    paste.to_excel(w, sheet_name="Для Excel-модели", index=False)
    funnel.to_excel(w, sheet_name="Воронка отбора", index=False)
    cand.to_excel(w, sheet_name="Кандидаты", index=False)
    ours.to_excel(w, sheet_name="Наша пятёрка", index=False)
    quality.round(4).to_excel(w, sheet_name="Качество моделей")
    imp.sort_values(ascending=False).rename("Важность").to_excel(w, sheet_name="Важность признаков")
    vol_quality.round(4).to_excel(w, sheet_name="Волатильность методы")
    port.round(4).to_excel(w, sheet_name="Волатильность и веса")
    corr.round(3).to_excel(w, sheet_name="Корреляции")
    data.drop(columns=["index"], errors="ignore").to_excel(w, sheet_name="Все данные", index=False)
print("Сохранено: ml_results.xlsx и папка charts/")
print(paste.round(4).to_string(index=False))
try:
    from google.colab import files
    import shutil
    shutil.make_archive("charts", "zip", "charts")
    files.download("ml_results.xlsx"); files.download("charts.zip")
except Exception:
    pass

## Как читать результат
* **Качество моделей.** Если нейросеть не обогнала линейную регрессию или бустинг — так и пишем на слайде, а для отбора используем лучшую. Это сильнее, чем подгонять результат.
* **Кандидаты.** Модель видит только цифры. По каждому кандидату из топа нужно ответить: почему он дешёвый — временная проблема (кандидат) или структурная (ловушка).
* **Наша пятёрка.** Если модель тоже считает бумагу дешёвой — это второй независимый аргумент для слайда. Если нет — объясните, что модель упускает (например, разовый убыток или новая часть бизнеса).
* **Волатильность.** Вставьте значения с листа «Для Excel-модели» в указанные ячейки — веса и риск портфеля пересчитаются.